In [152]:
import seaborn as sns
df = sns.load_dataset('tips')

In [153]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [154]:
df['day'].unique()

['Sun', 'Sat', 'Thur', 'Fri']
Categories (4, object): ['Thur', 'Fri', 'Sat', 'Sun']

In [155]:
df.time.unique()

['Dinner', 'Lunch']
Categories (2, object): ['Lunch', 'Dinner']

In [156]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['time'] = encoder.fit_transform(df['time'])

In [157]:
df.time.unique()

array([0, 1])

In [158]:
# Divide data into dependent and independent features
X = df.drop(labels=['time'],axis=1)
y = df.time

In [159]:
X.head()

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4


In [160]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: time, dtype: int64

In [161]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test =train_test_split(X,y,test_size=0.20,random_state=42) 

In [162]:
# importing all important libraries for pipelining
from sklearn.pipeline import Pipeline # for pipelining
from sklearn.impute import SimpleImputer # for handeling missing values
from sklearn.preprocessing import StandardScaler # for feature scaling
from sklearn.preprocessing import OneHotEncoder # for categorical to numerical conversion
from sklearn.compose import ColumnTransformer # for automating various pipeline by combining them


In [163]:
categorical_col = ['sex','smoker','day']
numerical_col = ['total_bill','tip','size']

In [164]:
# Feature Engineering Automation
# Pipeline construction

# Numerical Pipeline
num_pipeline = Pipeline(
    steps = [
        ('imputer',SimpleImputer(strategy='median')), # Missing values will be replcaed with median
        ('scaler',StandardScaler())
    ]
)

# Categorical Pipeline
cat_pipeline = Pipeline(
    steps = [
        ('imputer',SimpleImputer(strategy='most_frequent')), # missing values will be replaced with mode
        ('onehotencoder',OneHotEncoder())
    ]
)

In [165]:
preprocessor = ColumnTransformer(
    [
        ('num_pipeline',num_pipeline,numerical_col),
        ('cat_pipeline',cat_pipeline,categorical_col)
    ]
)
preprocessor

,transformers,"[('num_pipeline', ...), ('cat_pipeline', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [166]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [186]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [188]:
# Model Training Automation

Models = {
    #'Random Forest': RandomForestClassifier(),
    'Logistic Regression' : LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'Decision Tree' : DecisionTreeClassifier(),
}

In [189]:
from sklearn.metrics import accuracy_score

In [190]:
# Function for Evaluation of different models

def evaluate_model(X_train,y_train,X_test,y_test,Model):

    report = {}
    for i in range(len(Model)):
        model = list(Model.values())[i]

        # Train model
        
        model.fit(X_train,y_train)

        # Predict Test Data

        y_pred = model.predict(X_test)

        # fetch accuracy score of model

        test_model_score = accuracy_score(y_test,y_pred)

        # fill the accuracy score in report

        report[list(Model.keys())[i]] =  test_model_score

        # return the report
    return report


In [191]:
evaluate_model(X_train,y_train,X_test,y_test,Models)

{'Logistic Regression': 1.0,
 'Random Forest': 0.9591836734693877,
 'Decision Tree': 0.9387755102040817}

In [192]:
classifier = RandomForestClassifier()

In [193]:
# Hyperparameter tunning

parameter = {
    'max_depth' :[3,5,10,None],
    'n_estimators' : [100,200,300,400],
    'criterion' : ['gini','entropy','log_loss']
}

In [194]:
from sklearn.model_selection import RandomizedSearchCV

In [195]:
cv = RandomizedSearchCV(classifier,param_distributions = parameter,scoring='accuracy',cv=5,verbose=3)

In [196]:
cv.fit(X_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END criterion=log_loss, max_depth=None, n_estimators=100;, score=0.974 total time=   0.4s
[CV 2/5] END criterion=log_loss, max_depth=None, n_estimators=100;, score=0.923 total time=   0.3s
[CV 3/5] END criterion=log_loss, max_depth=None, n_estimators=100;, score=0.974 total time=   0.4s
[CV 4/5] END criterion=log_loss, max_depth=None, n_estimators=100;, score=0.923 total time=   0.4s
[CV 5/5] END criterion=log_loss, max_depth=None, n_estimators=100;, score=0.923 total time=   0.4s
[CV 1/5] END criterion=log_loss, max_depth=None, n_estimators=200;, score=0.974 total time=   0.7s
[CV 2/5] END criterion=log_loss, max_depth=None, n_estimators=200;, score=0.923 total time=   0.8s
[CV 3/5] END criterion=log_loss, max_depth=None, n_estimators=200;, score=1.000 total time=   0.8s
[CV 4/5] END criterion=log_loss, max_depth=None, n_estimators=200;, score=0.949 total time=   0.8s
[CV 5/5] END criterion=log_loss, max_depth=None,

,estimator,RandomForestClassifier()
,param_distributions,"{'criterion': ['gini', 'entropy', ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...]}"
,n_iter,10
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [197]:
cv.best_params_

{'n_estimators': 200, 'max_depth': None, 'criterion': 'log_loss'}